In [1]:
# 29dklık kesintisiz parcaları birlestiriyor; 
# mid price, spread, spread bps hesaplıyor; 
# sigma dollar (volatility) ve kappa dollar (likidite yogunluğu) hesaplıyor

import pandas as pd
import numpy as np
import glob

# --- Veriyi yükle ---
trade_files = sorted(glob.glob("live_data/trades_*.parquet"))
book_files = sorted(glob.glob("live_data/book_*.parquet"))

trades_df = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
book_df = pd.concat([pd.read_parquet(f) for f in book_files], ignore_index=True)

trades_df = trades_df.sort_values('timestamp').reset_index(drop=True)
book_df = book_df.sort_values('timestamp').reset_index(drop=True)
trades_df['datetime'] = pd.to_datetime(trades_df['timestamp'], unit='ms')

# --- Kesintisiz (29 dakikalık) parçayı ayır ---
gaps = trades_df['datetime'].diff().sort_values(ascending=False)
gap_point = gaps.index[0]
split_time = trades_df.loc[gap_point, 'datetime']

part1 = trades_df[trades_df['datetime'] < split_time]
trades_df = part1.copy()
book_df = book_df[book_df['timestamp'] < int(split_time.timestamp() * 1000)].copy()

# --- mid price / spread ---
book_df = book_df.sort_values('timestamp').reset_index(drop=True)
book_df['mid_price'] = (book_df['bid_0_price'] + book_df['ask_0_price']) / 2
book_df['spread'] = book_df['ask_0_price'] - book_df['bid_0_price']
book_df['spread_bps'] = (book_df['spread'] / book_df['mid_price']) * 10000

# --- Volatilite (dolar cinsinden) ---
avg_dt_sec = book_df['timestamp'].diff().mean() / 1000
book_df['price_diff'] = book_df['mid_price'].diff()
sigma_dollar = book_df['price_diff'].std() / np.sqrt(avg_dt_sec)

# --- Kappa (dolar cinsinden) ---
book_sorted = book_df.sort_values('timestamp')
trades_sorted = trades_df.sort_values('timestamp')
merged = pd.merge_asof(trades_sorted, book_sorted[['timestamp', 'mid_price']], on='timestamp', direction='backward')
merged['distance_dollar'] = (merged['price'] - merged['mid_price']).abs()
avg_distance_dollar = merged['distance_dollar'].mean()
kappa_dollar = 1 / avg_distance_dollar

print(f"Toplam trade: {len(trades_df)}, toplam book update: {len(book_df)}")
print(f"Sigma (dolar): {sigma_dollar:.6f}, Kappa (dolar): {kappa_dollar:.6f}")

Toplam trade: 96008, toplam book update: 15838
Sigma (dolar): 4.159822, Kappa (dolar): 0.181661


In [ ]:
# offline gamma=0.01 taraması
# latency 0'dan 1000ms'e çıktıkça PnL +1.40'tan -69.49'a düşüyor

def run_avellaneda_stoikov_latency(gamma, sigma, kappa, latency_ms=0, order_size=0.01, max_inventory=0.2):
    inventory = 0.0
    cash = 0.0
    fills = []

    total_duration_sec = (book_df['timestamp'].iloc[-1] - book_df['timestamp'].iloc[0]) / 1000

    for i in range(len(book_df) - 1):
        row = book_df.iloc[i]
        next_ts = book_df.iloc[i + 1]['timestamp']
        mid = row['mid_price']

        elapsed_sec = (row['timestamp'] - book_df['timestamp'].iloc[0]) / 1000
        remaining_time = max(total_duration_sec - elapsed_sec, 1)

        reservation_price = mid - inventory * gamma * (sigma**2) * remaining_time
        optimal_spread = gamma * (sigma**2) * remaining_time + (2/gamma) * np.log(1 + gamma/kappa)

        our_bid = reservation_price - optimal_spread / 2
        our_ask = reservation_price + optimal_spread / 2

        effective_ts = row['timestamp'] + latency_ms

        window_trades = trades_df[
            (trades_df['timestamp'] >= effective_ts) &
            (trades_df['timestamp'] < next_ts + latency_ms)
        ]

        filled_this_window = False
        for _, t in window_trades.iterrows():
            if filled_this_window:
                break
            if t['side'] == 'sell' and t['price'] <= our_bid and inventory < max_inventory:
                inventory += order_size
                cash -= order_size * our_bid
                fills.append({'side': 'buy'})
                filled_this_window = True
            elif t['side'] == 'buy' and t['price'] >= our_ask and inventory > -max_inventory:
                inventory -= order_size
                cash += order_size * our_ask
                fills.append({'side': 'sell'})
                filled_this_window = True

    current_mid = book_df.iloc[-1]['mid_price']
    pnl = cash + inventory * current_mid
    buy_count = sum(1 for f in fills if f['side'] == 'buy')
    sell_count = sum(1 for f in fills if f['side'] == 'sell')

    return {
        'latency_ms': latency_ms,
        'total_fills': len(fills),
        'buy_fills': buy_count,
        'sell_fills': sell_count,
        'final_inventory': inventory,
        'cash': cash,
        'pnl': pnl
    }

best_gamma = 0.01
latency_values = [0, 10, 50, 100, 250, 500, 1000]
latency_results = [run_avellaneda_stoikov_latency(best_gamma, sigma_dollar, kappa_dollar, l) for l in latency_values]
latency_df = pd.DataFrame(latency_results)
latency_df

,latency_ms,total_fills,buy_fills,sell_fills,final_inventory,cash,pnl
0,0,38,15,23,-0.08,5173.509307,1.401307
1,10,41,16,25,-0.09,5820.743811,2.122311
2,50,51,20,31,-0.11,7111.818022,0.169522
3,100,73,29,44,-0.15,9692.875500,-4.827000
4,250,115,54,61,-0.07,4508.175469,-17.419031
5,500,206,113,93,0.20,-12978.783403,-48.513403
6,1000,290,155,135,0.20,-12999.759271,-69.489271


In [ ]:
# Gerçek round-trip latency ölçüldü: ~435ms medyan (İstanbul → Binance).

import time
import requests
import numpy as np

latencies = []

for _ in range(50):
    start = time.time()
    response = requests.get("https://fapi.binance.com/fapi/v1/time")
    end = time.time()
    latencies.append((end - start) * 1000)  # milisaniyeye çevir

latencies = np.array(latencies)

print(f"Ortalama latency: {latencies.mean():.2f} ms")
print(f"Medyan latency: {np.median(latencies):.2f} ms")
print(f"Min/Max: {latencies.min():.2f} / {latencies.max():.2f} ms")
print(f"Std sapma: {latencies.std():.2f} ms")

Ortalama latency: 434.97 ms
Medyan latency: 417.94 ms
Min/Max: 388.91 / 925.26 ms
Std sapma: 76.65 ms


In [ ]:
# gamma=0.01 + gerçek 435ms latency
# 169 fill, PnL -41.38 USDT

real_latency_result = run_avellaneda_stoikov_latency(
    gamma=0.01,
    sigma=sigma_dollar,
    kappa=kappa_dollar,
    latency_ms=435  # gerçek ölçülen medyan
)
print(real_latency_result)

{'latency_ms': 435, 'total_fills': 169, 'buy_fills': 90, 'sell_fills': 79, 'final_inventory': 0.10999999999999999, 'cash': np.float64(-7153.024781876758), 'pnl': np.float64(-41.376281876759094)}


In [ ]:
# gamma=0.01 ile event-driven, latency'siz market-making sistemi
# 15 min: Son envanter: 0.0700 BTC, Nakit: -4509.65 USDT, Toplam fill: 215

import websocket
import json
import time
import numpy as np

# --- Model parametreleri ---
GAMMA = 0.01
SIGMA = sigma_dollar
KAPPA = kappa_dollar
ORDER_SIZE = 0.01
MAX_INVENTORY = 0.2
SESSION_DURATION_SEC = 120  # 2 dakikalık ufuk

# --- Canlı durum (state) ---
state = {
    'inventory': 0.0,
    'cash': 0.0,
    'our_bid': None,
    'our_ask': None,
    'session_start': None,
    'fills': [],
    'msg_count': 0,
    'bid_filled': False,
    'ask_filled': False
}

def compute_quotes(mid_price):
    elapsed = time.time() - state['session_start']
    remaining_time = max(SESSION_DURATION_SEC - elapsed, 1)

    reservation_price = mid_price - state['inventory'] * GAMMA * (SIGMA**2) * remaining_time
    optimal_spread = GAMMA * (SIGMA**2) * remaining_time + (2/GAMMA) * np.log(1 + GAMMA/KAPPA)

    state['our_bid'] = reservation_price - optimal_spread / 2
    state['our_ask'] = reservation_price + optimal_spread / 2

def on_depth_update(data):
    state['msg_count'] += 1
    if state['msg_count'] % 100 == 0:
        print(f"Mesaj alındı: {state['msg_count']}, mevcut bid/ask: {state['our_bid']}, {state['our_ask']}")

    bid = float(data['b'][0][0])
    ask = float(data['a'][0][0])
    mid = (bid + ask) / 2
    compute_quotes(mid)

    # Yeni quote geldi, tekrar doldurulabilir
    state['bid_filled'] = False
    state['ask_filled'] = False

def on_trade(data):
    if state['our_bid'] is None or state['our_ask'] is None:
        return

    price = float(data['p'])
    if price == 0:
        return

    is_sell_side_taker = data['m']  # True -> satıcı taker

    if is_sell_side_taker and not state['bid_filled'] and price <= state['our_bid'] and state['inventory'] < MAX_INVENTORY:
        state['inventory'] += ORDER_SIZE
        state['cash'] -= ORDER_SIZE * state['our_bid']
        state['fills'].append({'ts': time.time(), 'side': 'buy', 'price': state['our_bid'], 'inventory': state['inventory']})
        state['bid_filled'] = True
        print(f"FILL (buy) @ {state['our_bid']:.2f} | inventory: {state['inventory']:.4f}")

    elif not is_sell_side_taker and not state['ask_filled'] and price >= state['our_ask'] and state['inventory'] > -MAX_INVENTORY:
        state['inventory'] -= ORDER_SIZE
        state['cash'] += ORDER_SIZE * state['our_ask']
        state['fills'].append({'ts': time.time(), 'side': 'sell', 'price': state['our_ask'], 'inventory': state['inventory']})
        state['ask_filled'] = True
        print(f"FILL (sell) @ {state['our_ask']:.2f} | inventory: {state['inventory']:.4f}")

def on_message(ws, message):
    msg = json.loads(message)
    stream = msg['stream']
    data = msg['data']

    if stream.endswith('@trade'):
        on_trade(data)
    elif 'depth' in stream:
        on_depth_update(data)

def on_open(ws):
    state['session_start'] = time.time()
    print("Bağlantı açıldı, event-driven market making başladı!")

def on_error(ws, error):
    print(f"Hata: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Bağlantı kapandı.")
    print(f"Son envanter: {state['inventory']:.4f} BTC, Nakit: {state['cash']:.2f} USDT")
    print(f"Toplam fill: {len(state['fills'])}")

streams = "btcusdt@trade/btcusdt@depth20@100ms"
url = f"wss://fstream.binance.com/stream?streams={streams}"

ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    on_error=on_error,
    on_close=on_close
)
ws.run_forever()

Bağlantı açıldı, event-driven market making başladı!
Mesaj alındı: 100, mevcut bid/ask: 64252.08556207285, 64281.81443792715
Mesaj alındı: 200, mevcut bid/ask: 64236.07553156869, 64264.024468431315
Mesaj alındı: 300, mevcut bid/ask: 64236.96914411444, 64263.13085588557
Mesaj alındı: 400, mevcut bid/ask: 64256.15693598208, 64280.54306401793
Mesaj alındı: 500, mevcut bid/ask: 64278.64598249327, 64301.254017506726
Mesaj alındı: 600, mevcut bid/ask: 64286.634176959946, 64307.46582304006
Mesaj alındı: 700, mevcut bid/ask: 64288.42395723402, 64307.47604276597
FILL (sell) @ 64319.40 | inventory: -0.0100
Mesaj alındı: 800, mevcut bid/ask: 64306.97911038534, 64324.2520037461
FILL (sell) @ 64316.54 | inventory: -0.0200
Mesaj alındı: 900, mevcut bid/ask: 64301.05057252219, 64316.54497172611
Mesaj alındı: 1000, mevcut bid/ask: 64301.951233821004, 64315.668780577704
FILL (sell) @ 64315.64 | inventory: -0.0300
Mesaj alındı: 1100, mevcut bid/ask: 64317.513721553616, 64329.460025474276
FILL (buy) @ 64

True

In [ ]:
# 215 fill, -11.04 USDT PnL

current_mid = (float(data['b'][0][0]) + float(data['a'][0][0])) / 2 if 'data' in dir() else state['our_bid']  # fallback

# Daha güvenilir yol: son bilinen mid'i quote fonksiyonundan al
final_mid = (state['our_bid'] + state['our_ask']) / 2  # yaklaşık son mid
final_pnl = state['cash'] + state['inventory'] * final_mid

print(f"Toplam fill: {len(state['fills'])}")
print(f"Son envanter: {state['inventory']:.4f} BTC")
print(f"Nakit: {state['cash']:.2f} USDT")
print(f"Yaklaşık son mid: {final_mid:.2f}")
print(f"Toplam PnL (mark-to-market): {final_pnl:.2f} USDT")

Toplam fill: 215
Son envanter: 0.0700 BTC
Nakit: -4509.65 USDT
Yaklaşık son mid: 64265.74
Toplam PnL (mark-to-market): -11.04 USDT


In [ ]:
# GAMMA=0.01 ile gerçek zamanlı + 435ms latency simülasyonlu event-driven versiyon
# gerçek round-trip gecikmenin canlı sistemde PnL'i nasıl kötüleştirdiğini gösterir — 683 fill, PnL -47.18 USDT

import websocket
import json
import time
import numpy as np
from collections import deque

# --- Model parametreleri ---
GAMMA = 0.01
SIGMA = sigma_dollar
KAPPA = kappa_dollar
ORDER_SIZE = 0.01
MAX_INVENTORY = 0.2
SESSION_DURATION_SEC = 120
LATENCY_SEC = 0.435  # gerçek ölçülen ~435ms

state = {
    'inventory': 0.0,
    'cash': 0.0,
    'session_start': None,
    'fills': [],
    'msg_count': 0,
    'bid_filled': False,
    'ask_filled': False,
    'quote_history': deque(maxlen=2000)  # (timestamp, bid, ask) geçmişi
}

def compute_quotes(mid_price):
    elapsed = time.time() - state['session_start']
    remaining_time = max(SESSION_DURATION_SEC - elapsed, 1)

    reservation_price = mid_price - state['inventory'] * GAMMA * (SIGMA**2) * remaining_time
    optimal_spread = GAMMA * (SIGMA**2) * remaining_time + (2/GAMMA) * np.log(1 + GAMMA/KAPPA)

    bid = reservation_price - optimal_spread / 2
    ask = reservation_price + optimal_spread / 2
    state['quote_history'].append((time.time(), bid, ask))

def get_delayed_quote():
    """Şu an geçerli sayılan quote, aslında LATENCY_SEC kadar ESKİ olan quote."""
    target_time = time.time() - LATENCY_SEC
    delayed = None
    for ts, bid, ask in state['quote_history']:
        if ts <= target_time:
            delayed = (bid, ask)
        else:
            break
    return delayed

def on_depth_update(data):
    state['msg_count'] += 1
    if state['msg_count'] % 200 == 0:
        print(f"Mesaj: {state['msg_count']}")

    bid = float(data['b'][0][0])
    ask = float(data['a'][0][0])
    mid = (bid + ask) / 2
    compute_quotes(mid)
    state['bid_filled'] = False
    state['ask_filled'] = False

def on_trade(data):
    delayed_quote = get_delayed_quote()
    if delayed_quote is None:
        return
    our_bid, our_ask = delayed_quote

    price = float(data['p'])
    if price == 0:
        return

    is_sell_side_taker = data['m']

    if is_sell_side_taker and not state['bid_filled'] and price <= our_bid and state['inventory'] < MAX_INVENTORY:
        state['inventory'] += ORDER_SIZE
        state['cash'] -= ORDER_SIZE * our_bid
        state['fills'].append({'ts': time.time(), 'side': 'buy', 'price': our_bid, 'inventory': state['inventory']})
        state['bid_filled'] = True
        print(f"FILL (buy) @ {our_bid:.2f} | inventory: {state['inventory']:.4f}")

    elif not is_sell_side_taker and not state['ask_filled'] and price >= our_ask and state['inventory'] > -MAX_INVENTORY:
        state['inventory'] -= ORDER_SIZE
        state['cash'] += ORDER_SIZE * our_ask
        state['fills'].append({'ts': time.time(), 'side': 'sell', 'price': our_ask, 'inventory': state['inventory']})
        state['ask_filled'] = True
        print(f"FILL (sell) @ {our_ask:.2f} | inventory: {state['inventory']:.4f}")

def on_message(ws, message):
    msg = json.loads(message)
    stream = msg['stream']
    data = msg['data']
    if stream.endswith('@trade'):
        on_trade(data)
    elif 'depth' in stream:
        on_depth_update(data)

def on_open(ws):
    state['session_start'] = time.time()
    print(f"Bağlantı açıldı, {LATENCY_SEC*1000:.0f}ms latency simülasyonuyla market making başladı!")

def on_error(ws, error):
    print(f"Hata: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Bağlantı kapandı.")
    if state['quote_history']:
        _, last_bid, last_ask = state['quote_history'][-1]
        final_mid = (last_bid + last_ask) / 2
        final_pnl = state['cash'] + state['inventory'] * final_mid
        print(f"Son envanter: {state['inventory']:.4f} BTC, Nakit: {state['cash']:.2f} USDT")
        print(f"Toplam fill: {len(state['fills'])}, PnL: {final_pnl:.2f} USDT")

streams = "btcusdt@trade/btcusdt@depth20@100ms"
url = f"wss://fstream.binance.com/stream?streams={streams}"

ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    on_error=on_error,
    on_close=on_close
)
ws.run_forever()

Bağlantı açıldı, 435ms latency simülasyonuyla market making başladı!
FILL (buy) @ 64066.84 | inventory: 0.0100
FILL (buy) @ 64066.84 | inventory: 0.0200
FILL (buy) @ 64066.86 | inventory: 0.0300
FILL (buy) @ 64066.86 | inventory: 0.0400
FILL (buy) @ 64060.17 | inventory: 0.0500
FILL (buy) @ 64060.17 | inventory: 0.0600
FILL (buy) @ 64056.98 | inventory: 0.0700
FILL (buy) @ 64056.99 | inventory: 0.0800
FILL (buy) @ 64037.46 | inventory: 0.0900
Mesaj: 200
FILL (sell) @ 64064.59 | inventory: 0.0800
FILL (sell) @ 64064.59 | inventory: 0.0700
FILL (sell) @ 64064.58 | inventory: 0.0600
FILL (sell) @ 64073.06 | inventory: 0.0500
FILL (sell) @ 64088.07 | inventory: 0.0400
FILL (sell) @ 64112.77 | inventory: 0.0300
FILL (sell) @ 64112.77 | inventory: 0.0200
FILL (sell) @ 64112.75 | inventory: 0.0100
FILL (sell) @ 64112.75 | inventory: -0.0000
Mesaj: 400
FILL (sell) @ 64129.23 | inventory: -0.0100
FILL (sell) @ 64129.21 | inventory: -0.0200
FILL (sell) @ 64129.21 | inventory: -0.0300
FILL (sell)

True

In [ ]:
# düşük gamma'da (0.001) latency arttıkça PnL hızla kötüleşiyor (-34→-86)
# ama yüksek gamma'da (0.1) latency arttıkça PnL aslında iyileşiyor (+0.62→+3.34)
# geniş/temkinli spread gecikmeyi telafi ediyor

def run_avellaneda_stoikov_latency(gamma, sigma, kappa, latency_ms, order_size=0.01, max_inventory=0.2):
    inventory = 0.0
    cash = 0.0
    fills = []

    total_duration_sec = (book_df['timestamp'].iloc[-1] - book_df['timestamp'].iloc[0]) / 1000

    for i in range(len(book_df) - 1):
        row = book_df.iloc[i]
        next_ts = book_df.iloc[i + 1]['timestamp']
        mid = row['mid_price']

        elapsed_sec = (row['timestamp'] - book_df['timestamp'].iloc[0]) / 1000
        remaining_time = max(total_duration_sec - elapsed_sec, 1)

        reservation_price = mid - inventory * gamma * (sigma**2) * remaining_time
        optimal_spread = gamma * (sigma**2) * remaining_time + (2/gamma) * np.log(1 + gamma/kappa)

        our_bid = reservation_price - optimal_spread / 2
        our_ask = reservation_price + optimal_spread / 2

        effective_ts = row['timestamp'] + latency_ms

        window_trades = trades_df[
            (trades_df['timestamp'] >= effective_ts) &
            (trades_df['timestamp'] < next_ts + latency_ms)
        ]

        filled_this_window = False
        for _, t in window_trades.iterrows():
            if filled_this_window:
                break
            if t['side'] == 'sell' and t['price'] <= our_bid and inventory < max_inventory:
                inventory += order_size
                cash -= order_size * our_bid
                fills.append({'side': 'buy'})
                filled_this_window = True
            elif t['side'] == 'buy' and t['price'] >= our_ask and inventory > -max_inventory:
                inventory -= order_size
                cash += order_size * our_ask
                fills.append({'side': 'sell'})
                filled_this_window = True

    current_mid = book_df.iloc[-1]['mid_price']
    pnl = cash + inventory * current_mid
    buy_count = sum(1 for f in fills if f['side'] == 'buy')
    sell_count = sum(1 for f in fills if f['side'] == 'sell')

    return {
        'gamma': gamma,
        'latency_ms': latency_ms,
        'total_fills': len(fills),
        'final_inventory': inventory,
        'pnl': pnl
    }

# Gamma x Latency grid taraması
gamma_values = [0.001, 0.005, 0.01, 0.05, 0.1]
latency_values = [0, 100, 435, 1000]

grid_results = []
for g in gamma_values:
    for l in latency_values:
        grid_results.append(run_avellaneda_stoikov_latency(g, sigma_dollar, kappa_dollar, l))

grid_df = pd.DataFrame(grid_results)
pivot = grid_df.pivot(index='gamma', columns='latency_ms', values='pnl')
pivot

latency_ms,0,100,435,1000
gamma,,,,
0.001,-33.905749,-56.983296,-69.309548,-86.358829
0.005,-6.118543,-21.533479,-54.268761,-67.127802
0.010,1.401307,-4.827000,-41.376282,-69.489271
0.050,0.622559,1.939413,-1.736415,-12.293365
0.100,0.621008,1.311976,2.325591,3.342585


In [ ]:
# her latency seviyesi için PnL'i maksimize eden optimal gamma
# latency 0/100/435/1000ms'de optimal gamma sırasıyla ~0.08/0.07/0.09/0.09
# gecikme ne olursa olsun en iyi strateji hep yüksek/temkinli gamma bölgesi

def optimize_gamma_for_latency(latency_ms, gamma_range, sigma=sigma_dollar, kappa=kappa_dollar):
    results = []
    for g in gamma_range:
        res = run_avellaneda_stoikov_latency(g, sigma, kappa, latency_ms)
        results.append(res)
    df = pd.DataFrame(results)
    best_row = df.loc[df['pnl'].idxmax()]
    return best_row, df

# Daha ince bir gamma aralığı (log ölçekte)
gamma_range = np.concatenate([
    np.arange(0.001, 0.01, 0.001),
    np.arange(0.01, 0.1, 0.01),
    np.arange(0.1, 0.5, 0.05)
])

latency_values = [0, 100, 435, 1000]
optimal_results = []

for l in latency_values:
    best_row, _ = optimize_gamma_for_latency(l, gamma_range)
    optimal_results.append(best_row)

optimal_df = pd.DataFrame(optimal_results)
optimal_df

,gamma,latency_ms,total_fills,final_inventory,pnl
16,0.08,0.0,7.0,-0.03,2.500410
15,0.07,100.0,15.0,-0.05,4.544535
17,0.09,435.0,24.0,0.06,4.443378
17,0.09,1000.0,37.0,0.17,3.454236


In [ ]:
# GAMMA=0.09 (optimizasyondan bulunan en iyi değer) + 435ms gerçek latency simülasyonu ile çalışan event-driven versiyon
# 174 fill, envanter -0.20/+0.02 aralığında dolaşıp uçlara yapışmadı
# Nakit -1324.40 USDT, PnL -7.54 USDT

import websocket
import json
import time
import numpy as np
from collections import deque

# --- Model parametreleri ---
GAMMA = 0.09  # optimizasyondan bulunan en iyi değer
SIGMA = sigma_dollar
KAPPA = kappa_dollar
ORDER_SIZE = 0.01
MAX_INVENTORY = 0.2
SESSION_DURATION_SEC = 120
LATENCY_SEC = 0.435  # gerçek ölçülen ~435ms

state = {
    'inventory': 0.0,
    'cash': 0.0,
    'session_start': None,
    'fills': [],
    'msg_count': 0,
    'bid_filled': False,
    'ask_filled': False,
    'quote_history': deque(maxlen=2000)
}

def compute_quotes(mid_price):
    elapsed = time.time() - state['session_start']
    remaining_time = max(SESSION_DURATION_SEC - elapsed, 1)

    reservation_price = mid_price - state['inventory'] * GAMMA * (SIGMA**2) * remaining_time
    optimal_spread = GAMMA * (SIGMA**2) * remaining_time + (2/GAMMA) * np.log(1 + GAMMA/KAPPA)

    bid = reservation_price - optimal_spread / 2
    ask = reservation_price + optimal_spread / 2
    state['quote_history'].append((time.time(), bid, ask))

def get_delayed_quote():
    target_time = time.time() - LATENCY_SEC
    delayed = None
    for ts, bid, ask in state['quote_history']:
        if ts <= target_time:
            delayed = (bid, ask)
        else:
            break
    return delayed

def on_depth_update(data):
    state['msg_count'] += 1
    if state['msg_count'] % 200 == 0:
        print(f"Mesaj: {state['msg_count']}")

    bid = float(data['b'][0][0])
    ask = float(data['a'][0][0])
    mid = (bid + ask) / 2
    compute_quotes(mid)
    state['bid_filled'] = False
    state['ask_filled'] = False

def on_trade(data):
    delayed_quote = get_delayed_quote()
    if delayed_quote is None:
        return
    our_bid, our_ask = delayed_quote

    price = float(data['p'])
    if price == 0:
        return

    is_sell_side_taker = data['m']

    if is_sell_side_taker and not state['bid_filled'] and price <= our_bid and state['inventory'] < MAX_INVENTORY:
        state['inventory'] += ORDER_SIZE
        state['cash'] -= ORDER_SIZE * our_bid
        state['fills'].append({'ts': time.time(), 'side': 'buy', 'price': our_bid, 'inventory': state['inventory']})
        state['bid_filled'] = True
        print(f"FILL (buy) @ {our_bid:.2f} | inventory: {state['inventory']:.4f}")

    elif not is_sell_side_taker and not state['ask_filled'] and price >= our_ask and state['inventory'] > -MAX_INVENTORY:
        state['inventory'] -= ORDER_SIZE
        state['cash'] += ORDER_SIZE * our_ask
        state['fills'].append({'ts': time.time(), 'side': 'sell', 'price': our_ask, 'inventory': state['inventory']})
        state['ask_filled'] = True
        print(f"FILL (sell) @ {our_ask:.2f} | inventory: {state['inventory']:.4f}")

def on_message(ws, message):
    msg = json.loads(message)
    stream = msg['stream']
    data = msg['data']
    if stream.endswith('@trade'):
        on_trade(data)
    elif 'depth' in stream:
        on_depth_update(data)

def on_open(ws):
    state['session_start'] = time.time()
    print(f"Bağlantı açıldı, gamma={GAMMA}, {LATENCY_SEC*1000:.0f}ms latency ile market making başladı!")

def on_error(ws, error):
    print(f"Hata: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Bağlantı kapandı.")
    if state['quote_history']:
        _, last_bid, last_ask = state['quote_history'][-1]
        final_mid = (last_bid + last_ask) / 2
        final_pnl = state['cash'] + state['inventory'] * final_mid
        print(f"Son envanter: {state['inventory']:.4f} BTC, Nakit: {state['cash']:.2f} USDT")
        print(f"Toplam fill: {len(state['fills'])}, PnL: {final_pnl:.2f} USDT")

streams = "btcusdt@trade/btcusdt@depth20@100ms"
url = f"wss://fstream.binance.com/stream?streams={streams}"

ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    on_error=on_error,
    on_close=on_close
)
ws.run_forever()

Bağlantı açıldı, gamma=0.09, 435ms latency ile market making başladı!
Mesaj: 200
Mesaj: 400
Mesaj: 600
Mesaj: 800
FILL (sell) @ 64414.29 | inventory: -0.0100
FILL (sell) @ 64414.21 | inventory: -0.0200
FILL (sell) @ 64417.03 | inventory: -0.0300
FILL (sell) @ 64416.95 | inventory: -0.0400
FILL (sell) @ 64421.76 | inventory: -0.0500
FILL (buy) @ 64394.26 | inventory: -0.0400
FILL (buy) @ 64394.26 | inventory: -0.0300
FILL (buy) @ 64394.33 | inventory: -0.0200
Mesaj: 1000
FILL (buy) @ 64404.77 | inventory: -0.0100
FILL (buy) @ 64405.00 | inventory: 0.0000
FILL (buy) @ 64405.00 | inventory: 0.0100
FILL (sell) @ 64402.98 | inventory: 0.0000
FILL (sell) @ 64402.98 | inventory: -0.0100
FILL (sell) @ 64404.98 | inventory: -0.0200
FILL (sell) @ 64405.38 | inventory: -0.0300
Mesaj: 1200
FILL (sell) @ 64410.75 | inventory: -0.0400
FILL (sell) @ 64410.75 | inventory: -0.0500
FILL (sell) @ 64424.68 | inventory: -0.0600
FILL (sell) @ 64433.09 | inventory: -0.0700
FILL (sell) @ 64433.09 | inventory:

True

In [1]:
# Market making'te envanter riski ile kâr arasında yapısal bir gerilim var
# Latency, bu dengeyi ciddi şekilde bozan ek bir faktör 
# ama geniş/temkinli spread stratejisi (yüksek gamma) latency dezavantajını kısmen telafi edebiliyor.

In [2]:
import websocket
import json
import time
import numpy as np
from collections import deque

# --- Model parametreleri ---
GAMMA = 0.09
SIGMA = sigma_dollar
KAPPA = kappa_dollar
ORDER_SIZE = 0.01
MAX_INVENTORY = 0.2
SESSION_DURATION_SEC = 120

# Rastgele latency parametreleri (gerçek ölçümden)
LATENCY_MEAN_SEC = 0.425
LATENCY_STD_SEC = 0.070

state = {
    'inventory': 0.0,
    'cash': 0.0,
    'session_start': None,
    'fills': [],
    'msg_count': 0,
    'bid_filled': False,
    'ask_filled': False,
    'quote_history': deque(maxlen=2000)
}

def compute_quotes(mid_price):
    elapsed = time.time() - state['session_start']
    remaining_time = max(SESSION_DURATION_SEC - elapsed, 1)

    reservation_price = mid_price - state['inventory'] * GAMMA * (SIGMA**2) * remaining_time
    optimal_spread = GAMMA * (SIGMA**2) * remaining_time + (2/GAMMA) * np.log(1 + GAMMA/KAPPA)

    bid = reservation_price - optimal_spread / 2
    ask = reservation_price + optimal_spread / 2
    state['quote_history'].append((time.time(), bid, ask))

def get_delayed_quote():
    # Her çağrıda gerçekçi, değişken bir latency çek (negatif olmasın diye clip)
    sampled_latency = max(np.random.normal(LATENCY_MEAN_SEC, LATENCY_STD_SEC), 0.05)
    target_time = time.time() - sampled_latency
    delayed = None
    for ts, bid, ask in state['quote_history']:
        if ts <= target_time:
            delayed = (bid, ask)
        else:
            break
    return delayed

def on_depth_update(data):
    state['msg_count'] += 1
    if state['msg_count'] % 200 == 0:
        print(f"Mesaj: {state['msg_count']}")

    bid = float(data['b'][0][0])
    ask = float(data['a'][0][0])
    mid = (bid + ask) / 2
    compute_quotes(mid)
    state['bid_filled'] = False
    state['ask_filled'] = False

def on_trade(data):
    delayed_quote = get_delayed_quote()
    if delayed_quote is None:
        return
    our_bid, our_ask = delayed_quote

    price = float(data['p'])
    if price == 0:
        return

    is_sell_side_taker = data['m']

    if is_sell_side_taker and not state['bid_filled'] and price <= our_bid and state['inventory'] < MAX_INVENTORY:
        state['inventory'] += ORDER_SIZE
        state['cash'] -= ORDER_SIZE * our_bid
        state['fills'].append({'ts': time.time(), 'side': 'buy', 'price': our_bid, 'inventory': state['inventory']})
        state['bid_filled'] = True
        print(f"FILL (buy) @ {our_bid:.2f} | inventory: {state['inventory']:.4f}")

    elif not is_sell_side_taker and not state['ask_filled'] and price >= our_ask and state['inventory'] > -MAX_INVENTORY:
        state['inventory'] -= ORDER_SIZE
        state['cash'] += ORDER_SIZE * our_ask
        state['fills'].append({'ts': time.time(), 'side': 'sell', 'price': our_ask, 'inventory': state['inventory']})
        state['ask_filled'] = True
        print(f"FILL (sell) @ {our_ask:.2f} | inventory: {state['inventory']:.4f}")

def on_message(ws, message):
    msg = json.loads(message)
    stream = msg['stream']
    data = msg['data']
    if stream.endswith('@trade'):
        on_trade(data)
    elif 'depth' in stream:
        on_depth_update(data)

def on_open(ws):
    state['session_start'] = time.time()
    print(f"Bağlantı açıldı, gamma={GAMMA}, rastgele latency (mean={LATENCY_MEAN_SEC*1000:.0f}ms, std={LATENCY_STD_SEC*1000:.0f}ms) ile market making başladı!")

def on_error(ws, error):
    print(f"Hata: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Bağlantı kapandı.")
    if state['quote_history']:
        _, last_bid, last_ask = state['quote_history'][-1]
        final_mid = (last_bid + last_ask) / 2
        final_pnl = state['cash'] + state['inventory'] * final_mid
        print(f"Son envanter: {state['inventory']:.4f} BTC, Nakit: {state['cash']:.2f} USDT")
        print(f"Toplam fill: {len(state['fills'])}, PnL: {final_pnl:.2f} USDT")

streams = "btcusdt@trade/btcusdt@depth20@100ms"
url = f"wss://fstream.binance.com/stream?streams={streams}"

ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    on_error=on_error,
    on_close=on_close
)
ws.run_forever()

Bağlantı açıldı, gamma=0.09, rastgele latency (mean=425ms, std=70ms) ile market making başladı!
Mesaj: 200
Mesaj: 400
Mesaj: 600
Mesaj: 800
FILL (sell) @ 65897.86 | inventory: -0.0100
Mesaj: 1000
Mesaj: 1200
FILL (sell) @ 65892.52 | inventory: -0.0200
FILL (sell) @ 65892.72 | inventory: -0.0300
FILL (buy) @ 65879.85 | inventory: -0.0200
FILL (sell) @ 65884.33 | inventory: -0.0300
FILL (sell) @ 65884.33 | inventory: -0.0400
FILL (sell) @ 65884.33 | inventory: -0.0500
Mesaj: 1400
FILL (buy) @ 65875.38 | inventory: -0.0400
Mesaj: 1600
FILL (sell) @ 65882.16 | inventory: -0.0500
Mesaj: 1800
FILL (sell) @ 65890.28 | inventory: -0.0600
FILL (sell) @ 65896.79 | inventory: -0.0700
FILL (sell) @ 65896.79 | inventory: -0.0800
Mesaj: 2000
FILL (buy) @ 65895.72 | inventory: -0.0700
FILL (buy) @ 65895.72 | inventory: -0.0600
Mesaj: 2200
FILL (sell) @ 65899.89 | inventory: -0.0700
FILL (sell) @ 65906.91 | inventory: -0.0800
FILL (sell) @ 65906.91 | inventory: -0.0900
FILL (sell) @ 65906.91 | invento

True